In [1]:
from activitysim.core.random import Random

import pandas as pd
import numpy as np

In [2]:
prng = np.random.default_rng(seed=12345)
idxs = pd.Index(np.sort(prng.integers(1_000_000_000, size=250_000)))
idxs = idxs.unique()
print(f"Length of idxs: {idxs.size}")

Length of idxs: 249972


In [3]:
hh = pd.DataFrame(index=idxs, columns=[])
hh.index.name = "household_id"
hh["dummy"] = 1
hh

,dummy
household_id,
5176,1
6338,1
8523,1
11455,1
19127,1
...,...
999980188,1
999983015,1
999987465,1


In [4]:
r = Random(channel_type="faster")
r.set_base_seed(42)

In [5]:
r.add_channel("households", hh, fast=True)

In [6]:
shh = hh.copy()
shh.index.name = "slow_hhid"

In [7]:
r.add_channel("slow_households", shh, fast=False)

In [8]:
%%time

r.begin_step(step_name="peekaboo")

CPU times: user 1.02 ms, sys: 518 μs, total: 1.54 ms
Wall time: 983 μs


In [9]:
# first call per session to each randomization function on fast channel will trigger 
# numba compilation, which takes an extra second or two
# compilation is once per ActivitySim session, not once per step, so all subsequent 
# calls to [random_for_df, normal_for_df] will be faster

# the first call to any randomization function on a channel within a step will trigger reseeding, this happens once per step only
# but it does happen every new step.

r.random_for_df(hh)

array([[0.15779417],
       [0.53942147],
       [0.08538743],
       ...,
       [0.08571587],
       [0.70314741],
       [0.16105867]])

In [10]:
%%timeit -n 10

# subsequent calls to random_for_df on the same channel within the same step should be fast, no numba compilation, no reseeding

r.random_for_df(hh)

7.08 ms ± 35.2 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [11]:
%%time

# by contrast, the old slow channel will trigger reseeding every time

r.random_for_df(shh)

CPU times: user 422 ms, sys: 5.54 ms, total: 428 ms
Wall time: 427 ms


array([[0.12387598],
       [0.37916747],
       [0.11812594],
       ...,
       [0.22852176],
       [0.08680529],
       [0.97651523]])

In [12]:
%%timeit -n 3 -r 5

# so it is always slow

r.random_for_df(shh)

474 ms ± 4.91 ms per loop (mean ± std. dev. of 5 runs, 3 loops each)


In [13]:
## Normal distribution with parameters

In [14]:
## again, first call is slow (a bit) due to numba compilation

r.normal_for_df(hh, mu=3.0, sigma=1.5)

array([[3.80394339],
       [5.29739599],
       [0.44868825],
       ...,
       [1.09076021],
       [1.72714507],
       [3.78585531]])

In [15]:
%%timeit -n 10

# subsequent calls should be fast, no numba compilation, no reseeding

r.normal_for_df(hh, mu=3.0, sigma=1.5)

7.57 ms ± 56.7 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [16]:
%%timeit -n 3 -r 5

# old slow channel is slow every time due to reseeding for normals too

r.normal_for_df(shh, mu=3.0, sigma=1.5)

486 ms ± 13.6 ms per loop (mean ± std. dev. of 5 runs, 3 loops each)


In [17]:
# choice_for_df sets atop the random_for_df state, so it should be fast after the first call to random_for_df

r.choice_for_df(hh, a=[1,2,3,4,5,6,7], size=5, replace=False).reshape(-1, 5)

array([[1, 2, 3, 5, 4],
       [3, 2, 4, 5, 1],
       [1, 4, 3, 5, 2],
       ...,
       [5, 4, 7, 3, 2],
       [7, 3, 4, 1, 5],
       [2, 4, 5, 6, 3]])

In [18]:
# old way still slow

r.choice_for_df(shh, a=[1,2,3,4,5,6,7], size=5, replace=False)

array([3, 5, 4, ..., 3, 1, 5])

In [19]:
## Reproducible within the same named step, but different across steps due to reseeding

In [20]:
r.end_step("peekaboo")
r.begin_step("peekaboo")

In [21]:
%%time

# first new random draw after resetting the step, will need to reseed, so takes a moment
# but if using quick entropy, still faster than the old way

r.normal_for_df(hh, mu=3.0, sigma=1.5)

CPU times: user 286 ms, sys: 6.4 ms, total: 292 ms
Wall time: 296 ms


array([[ 2.22074798],
       [ 4.0869555 ],
       [-0.40208594],
       ...,
       [ 2.36546569],
       [ 2.33117172],
       [ 2.12748091]])

In [22]:
%%time

# old way not appreciably different, as reseeding happens every time we want to make any draw

r.normal_for_df(shh, mu=3.0, sigma=1.5)

CPU times: user 443 ms, sys: 6.14 ms, total: 449 ms
Wall time: 454 ms


array([1.48765918, 1.65064059, 2.49874109, ..., 2.84916606, 3.15987699,
       3.07499564])

In [23]:
r.end_step("peekaboo")
r.begin_step("peekaboo")

In [24]:
%%time

# after resetting the step with the same name, we recover the same random numbers again

r.normal_for_df(hh, mu=3.0, sigma=1.5)

CPU times: user 265 ms, sys: 5.16 ms, total: 270 ms
Wall time: 269 ms


array([[ 2.22074798],
       [ 4.0869555 ],
       [-0.40208594],
       ...,
       [ 2.36546569],
       [ 2.33117172],
       [ 2.12748091]])

In [25]:
%%time

r.normal_for_df(shh, mu=3.0, sigma=1.5)

CPU times: user 428 ms, sys: 3.67 ms, total: 432 ms
Wall time: 433 ms


array([1.48765918, 1.65064059, 2.49874109, ..., 2.84916606, 3.15987699,
       3.07499564])

In [26]:
# Faster is faster on a small subset also

In [27]:
hh3 = hh.iloc[2:5]
shh3 = shh.iloc[2:5]

In [28]:
%%timeit -n 3

r.normal_for_df(hh3, mu=3.0, sigma=1.5)

49 μs ± 21.9 μs per loop (mean ± std. dev. of 7 runs, 3 loops each)


In [29]:
%%timeit -n 3

r.normal_for_df(shh3, mu=3.0, sigma=1.5)

504 μs ± 121 μs per loop (mean ± std. dev. of 7 runs, 3 loops each)
